# 08 - Q3 first tranche: does Gemma track emotion transitions while reading?

**What this notebook is for.** [google/gemma-4-31b-it](https://huggingface.co/google/gemma-4-31b-it)
reads multi-phase stories token by token: each story was written so its emotional
tone moves through its tagged emotions one after another (all stories here are
SEQUENTIAL in the design's sense). We ask two plain questions about the
activations recorded during that reading, with every scoring convention
registered in TREE.md (node Q3.H1.E1) before any scoring ran. This extends the
probe-validation story of [notebook 03](03_detection_probe_campaign.ipynb) into
dynamics, on the emotion directions replicated from
[Anthropic's emotions paper](https://transformer-circuits.pub/2026/emotions)
([arXiv 2604.07729](https://arxiv.org/abs/2604.07729)).

- The *identity read* asks: while the model reads a story phase tagged with one
  emotion, does that emotion's probe outrank the rest of its bank? Its registry
  name in TREE.md is "gate read G", because the dynamics reads are conditional
  on it: if the current emotion cannot be read out at all, questions about
  transitions are moot.
- The *anticipation read* asks: does the incoming emotion's signal start rising
  in the 16 tokens just before the phase boundary, compared to the 16 tokens
  before those? Its registry name is "R1".

**Key concepts, plain words first.**

- An *emotion probe* is one emotion's direction in the model's residual stream:
  the mean activation while reading that emotion's stories, minus the mean over
  the whole emotion pool, unit-normalized. A *probe bank* is a set of probes
  read out together, named by who wrote the probe stories:
  the **corpus bank** (171 emotions, external stories), the **self-gen bank**
  (12 emotions, stories the probed model wrote itself; the detection winner of
  [notebook 07](07_generator_lineages.ipynb), registry E10), the **DeepSeek
  bank** (12 emotions, [DeepSeek-written stories](https://huggingface.co/datasets/abotresol/emotion-stories-deepseek-v4-pro);
  the E11 winner, added by a registered amendment and therefore present only on
  the newer arms), and **24 random directions**, the meaninglessness floor
  (registry name N1).
- *Centered cosine*: before taking the cosine between an activation and a
  probe, the story set's own mean activation is subtracted per (layer, probe).
  This is the registered scoring convention (the readout lesson of notebook 03).
- *Rank*: for one story phase, average each probe's centered cosine over the
  phase's tokens, then rank the tagged emotion among its bank; rank 1 means the
  correct probe scored highest. Chance median rank is about half the bank size.
- *Anticipation lead*: the incoming emotion's mean centered cosine over the
  W = 16 tokens just before the phase boundary, minus its mean over the 16
  tokens before those. Positive means the signal is already rising before the
  text switches. Leads are reported in units of the calibrated per-token noise
  ("noise-sd"), so different layers share one scale.
- *Three nulls, in plain words.* (1) **Wrong-emotion shuffles** (registry N2):
  reassign the emotion labels at random 10,000 times; this measures where
  chance sits instead of assuming it. (2) **Random directions** (N1): run the
  whole pipeline on 24 random vectors; this is what "the probes mean nothing"
  looks like. (3) The **constant-emotion control stories**: same story
  scaffold, same marked scene changes, but no emotion change; this tests
  whether "anticipation" is really just scene-change mechanics. The instrument
  calibration also measured the per-token noise floor, so a null result comes
  with its detection power attached.
- The *verdict ladder* (tiers T0 to T3) is the registered mapping from outcomes
  to allowed conclusions, written down before scoring (TREE Q3.H1.E1). No tier
  is claimed in this notebook.
- Registry codes (Q3.H1.E1, G, R1, N1, N2) are coordinates of nodes in TREE.md;
  prose here gives the plain meaning first and the code in parentheses so the
  registered version can be found.

**Index.**
1. Does the model know which emotion the current phase is? (identity read, gate G)
2. Stories or vectors: what does each result depend on? (the dissociation)
3. Above chance versus useful versus meaningless: the three-way diagnosis
4. The full design grid and the combined reading
5. What is still open

**The story sets and the evidence files.** Four scored arms; per-token probe
dots live in the trajectory datasets (probe identity in each arm's
`probe_labels.json`), and [DATA.md](../DATA.md) plus `ROUTES` in
`src/emotion_vectors/artifacts.py` map every local name to its dataset URL.

| story set (arm) | stories written by | probe banks carried | trajectory dataset |
|---|---|---|---|
| Gemma-written, v1 | gemma-4-31b-it | corpus-171 (pre-fix) + self-gen-12 + random-24 | [emotion-combined-trajectories-gemma-4-31b-it](https://huggingface.co/datasets/abotresol/emotion-combined-trajectories-gemma-4-31b-it) |
| Gemma-written, v2 | same stories, re-extracted | post-fix corpus-171 + self-gen-12 + DeepSeek-12 + random-24 | [emotion-combined-trajectories-gemma-4-31b-it-v2](https://huggingface.co/datasets/abotresol/emotion-combined-trajectories-gemma-4-31b-it-v2) |
| DeepSeek-written | DeepSeek v4 Pro | all three banks + random | [emotion-combined-trajectories-deepseek-stories-gemma-4-31b-it](https://huggingface.co/datasets/abotresol/emotion-combined-trajectories-deepseek-stories-gemma-4-31b-it) |
| constant-emotion control | DeepSeek v4 Pro | all three banks + random | [emotion-combined-trajectories-constant-control-gemma-4-31b-it](https://huggingface.co/datasets/abotresol/emotion-combined-trajectories-constant-control-gemma-4-31b-it) |

A fifth arm (the same Gemma-written stories read by the base model
[google/gemma-4-31b](https://huggingface.co/google/gemma-4-31b),
[trajectories here](https://huggingface.co/datasets/abotresol/emotion-combined-trajectories-gemma-4-31b)) is
collected but deliberately unscored: it is reserved for the registered falsify
plan. Evidence files scored from the arms, all in the
[experiment-artifacts dataset](https://huggingface.co/datasets/abotresol/emotion-vectors-experiment-artifacts): `q3_gate_r1_it.json` (v1 + control),
`q3_gate_r1_deepseek.json`, `q3_gate_r1_it_v2.json`.


In [1]:
# this cell loads the scored evidence files and defines shared constants.
# fetch() resolves each file under local results/ first, then the published
# Hugging Face dataset, so the notebook runs unchanged on any clone.
import json

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from emotion_vectors.artifacts import fetch

MODEL = "google/gemma-4-31b-it"
LAYERS = [6, 15, 24, 33, 42, 51]   # the six extracted layers
PRIMARY_LAYER = 33                 # the registered primary cell

def load_json(name):
    return json.loads(fetch(name).read_text())

gemma_stories = load_json("q3_gate_r1_it.json")           # Gemma-written stories, v1 probe shards (+ control arm)
deepseek_stories = load_json("q3_gate_r1_deepseek.json")  # DeepSeek-written stories
try:  # v2: the same Gemma stories re-extracted with post-fix corpus probes + the DeepSeek bank
    gemma_stories_v2 = load_json("q3_gate_r1_it_v2.json")
except FileNotFoundError:
    gemma_stories_v2 = None
    print("v2 evidence file absent: the v2 story-set column will be missing from every figure")

BANK_SIZE = {"corpus": 171, "selfgen": 12, "deepseek": 12}
BANK_LABEL = {
    "corpus": "corpus-171 probes",
    "selfgen": "self-gen probes (12)",
    "deepseek": "DeepSeek probes (12)",
}
SUBSTRATES = {
    "Gemma-written (v1 probes)": gemma_stories["true_arm"],
    "DeepSeek-written stories": deepseek_stories["true_arm"],
    "control (no emotion change)": gemma_stories["control_arm"],
}
if gemma_stories_v2 is not None:
    SUBSTRATES["Gemma-written (v2 probes)"] = gemma_stories_v2["true_arm"]
print("stories scored per arm:", {name: arm["n_stories"] for name, arm in SUBSTRATES.items()})


stories scored per arm: {'Gemma-written (v1 probes)': 2998, 'DeepSeek-written stories': 2835, 'control (no emotion change)': 106, 'Gemma-written (v2 probes)': 3010}


## 1. Does the model know which emotion the current phase is? (identity read, gate G)

**Answer: yes, clearly, when read with good probes.** On every story set, the
two 12-probe banks place the tagged emotion near the top of the bank: median
rank 1 to 6 of 12 at every layer (the self-gen bank everywhere; the DeepSeek
bank on the arms that carry it, since the v1 Gemma-written arm was extracted
before that third bank existed). The corpus-171 bank never comes close (median
rank 35 to 83 of 171). Both ranges are recomputed at runtime in the figure
title, and the printout below the figure gives them per bank and story set.

**How it was measured.** For every story phase, average each probe's centered
cosine over the phase's tokens, then rank the tagged emotion among its bank
(rank 1 = the correct probe scored highest). The figure divides rank by bank
size so banks of different sizes share one scale: chance sits near 0.5, the
registered usefulness bar is the top decile, 0.1. Lower is better.

**Verdict.** The identity read passes in substance for the self-gen and
DeepSeek banks: wrong-emotion shuffles (N2) essentially never match their ranks
(p < 0.001 at every layer on both Gemma-written arms; the exceptions cluster at
the last extracted layer, 51, on the DeepSeek-written and control arms, and on
the small 106-story control arm at layer 15; the printout gives each bank's
worst shuffle p and the layer it occurs at). The corpus-171 bank fails the
registered bar at every layer on every story set. Formal verdict-ladder
adjudication is pending because the registered gate text named the corpus-171
bank specifically (section 5, point 1).


In [2]:
# this cell plots the tagged emotion's median rank (as a fraction of bank
# size) per layer, probe bank, and story set, then prints each bank's rank
# range and its worst wrong-emotion-shuffle p value across the six layers
fig = make_subplots(rows=1, cols=len(SUBSTRATES), subplot_titles=list(SUBSTRATES),
                    shared_yaxes=True)
colors = {"corpus": "#7f7f7f", "selfgen": "#1f77b4", "deepseek": "#d62728"}
banks_in_legend = set()  # each bank gets exactly one legend entry, whichever panel shows it first
small_bank_ranks, corpus_ranks = [], []  # feed the title's computed rank ranges
for col, (substrate, arm) in enumerate(SUBSTRATES.items(), start=1):
    for bank, per_layer_stats in arm["gate_G"].items():
        if bank == "random" or not per_layer_stats.get("per_layer"):
            continue
        ranks = [per_layer_stats["per_layer"][str(layer)]["median_rank"] for layer in LAYERS]
        (corpus_ranks if bank == "corpus" else small_bank_ranks).extend(ranks)
        fig.add_bar(
            x=[str(layer) for layer in LAYERS],
            y=[rank / BANK_SIZE[bank] for rank in ranks],
            name=BANK_LABEL[bank], marker_color=colors[bank],
            legendgroup=bank, showlegend=(bank not in banks_in_legend), row=1, col=col,
        )
        banks_in_legend.add(bank)
# the two reference lines are labeled in the legend (per-panel text collided with bars)
fig.add_hline(y=0.5, line_dash="dot")
fig.add_hline(y=0.1, line_dash="dash")
fig.add_scatter(x=[None], y=[None], mode="lines", line=dict(dash="dot", color="#2a3f5f"),
                name="chance = 0.5 (N2 shuffle)")
fig.add_scatter(x=[None], y=[None], mode="lines", line=dict(dash="dash", color="#2a3f5f"),
                name="registered bar = 0.1 (top decile)")
for annotation in fig.layout.annotations:  # only the panel titles exist at this point
    annotation.font = dict(size=12)
fig.update_layout(
    title=(
        "Does the model know which emotion the current phase is? "
        "Yes with the 12-probe banks, no with the corpus bank<br>"
        f"<sup>tagged emotion's median rank: {min(small_bank_ranks):.0f} to "
        f"{max(small_bank_ranks):.0f} of 12 for the self-gen and DeepSeek banks, vs "
        f"{min(corpus_ranks):.0f} to {max(corpus_ranks):.0f} of 171 for the corpus bank</sup><br>"
        f"<sup>one bar = one (story set, probe bank, layer); identity read (gate G) | {MODEL}</sup>"
    ),
    title_font_size=15,
    yaxis_title="median rank / bank size (lower is better)",
    barmode="group", width=1400, height=470, margin=dict(t=120),
)
fig.update_xaxes(title="layer")
fig.show()
for substrate, arm in SUBSTRATES.items():
    for bank, per_layer_stats in arm["gate_G"].items():
        if bank == "random" or not per_layer_stats.get("per_layer"):
            continue
        cells = per_layer_stats["per_layer"]
        ranks = [cells[str(layer)]["median_rank"] for layer in LAYERS]
        worst_layer = max(LAYERS, key=lambda layer: cells[str(layer)]["n2_p"])
        print(f"{substrate:28s} {BANK_LABEL[bank]:22s} median rank {min(ranks):.0f} to "
              f"{max(ranks):.0f} of {BANK_SIZE[bank]}; worst shuffle p = "
              f"{cells[str(worst_layer)]['n2_p']:.4f} (layer {worst_layer})")


Gemma-written (v1 probes)    corpus-171 probes      median rank 38 to 76 of 171; worst shuffle p = 0.0000 (layer 6)
Gemma-written (v1 probes)    self-gen probes (12)   median rank 1 to 3 of 12; worst shuffle p = 0.0000 (layer 6)
DeepSeek-written stories     corpus-171 probes      median rank 55 to 83 of 171; worst shuffle p = 0.0028 (layer 51)
DeepSeek-written stories     self-gen probes (12)   median rank 3 to 6 of 12; worst shuffle p = 0.4833 (layer 51)
DeepSeek-written stories     DeepSeek probes (12)   median rank 2 to 5 of 12; worst shuffle p = 0.0001 (layer 51)
control (no emotion change)  corpus-171 probes      median rank 50 to 82 of 171; worst shuffle p = 0.2157 (layer 15)
control (no emotion change)  self-gen probes (12)   median rank 3 to 6 of 12; worst shuffle p = 0.4817 (layer 51)
control (no emotion change)  DeepSeek probes (12)   median rank 2 to 5 of 12; worst shuffle p = 0.0016 (layer 51)
Gemma-written (v2 probes)    corpus-171 probes      median rank 35 to 77 of 171; 

<details><summary><b>How to read this figure</b></summary>

Each panel is one story set. The x axis is the model layer the activations were
read at (six extracted layers). The y axis is the tagged emotion's median rank
divided by its bank size, so a 12-probe bank and the 171-probe bank share one
scale; lower means the correct emotion sits nearer the top of its bank. Bars
are colored by probe bank (legend at the right). The dotted line at 0.5 is
chance: where the median would land if emotion labels were assigned at random,
which the wrong-emotion shuffle (N2) confirms empirically. The dashed line at
0.1 is the registered usefulness bar: the tagged emotion in the top tenth of
its bank. A bar below the dashed line is doing good work. The printed lines
below give, per (story set, probe bank), the median-rank range across the six
layers and the worst shuffle p, which is the probability that 10,000
wrong-emotion shuffles produce a median rank at least that good. Numbers come
from `q3_gate_r1_it.json`, `q3_gate_r1_deepseek.json`, and
`q3_gate_r1_it_v2.json` in the
[experiment-artifacts dataset](https://huggingface.co/datasets/abotresol/emotion-vectors-experiment-artifacts).

</details>

**What section 1 means.** With well-lineaged 12-probe banks, the tagged
emotion is close to the top of the bank on every story set, including the
constant-emotion control (phase identity stays readable even when nothing
changes emotionally). The corpus-171 bank, the bank the gate was registered on,
stays near the middle of its bank everywhere. So whether the current emotion is
readable at all is decided by probe quality, not by who wrote the stories.
What would change this reading: the pending ladder adjudication going against
the substance reading (section 5), or the reserved base-model arm failing to
replicate it.


## 2. Stories or vectors: what does each result depend on?

**Answer: both, but for different claims.** Whether the emotional state is
VISIBLE at all depends on the vectors: on the same stories, corpus-171 probes
fail the identity read while self-gen and DeepSeek probes succeed. Whether
ANTICIPATION exists depends on the stories: the same probes show a solid
pre-boundary rise on Gemma-written stories and nothing on DeepSeek-written
ones.

**How it was measured.** Left panel: the identity read from section 1 (gate G),
as rank divided by bank size. Right panel: the anticipation read (R1), the
incoming emotion's rise over the 16 tokens before the phase boundary versus the
16 tokens before those, in units of the calibrated per-token noise (noise-sd).
Stars mark cells that pass their registered bar (identity: the top-decile gate
bar; anticipation: shuffle p < 0.001 plus the registered magnitude bar). The
slider below the figure scrubs all six layers; layer 33 is the registered
primary cell, and the dissociation pattern holds across the mid-to-late band.

**Verdict.** Vector quality decides whether the state is readable at all
(matching the detection results of
[notebook 07](07_generator_lineages.ipynb), E10/E11); the anticipation effect
is a property of the text. Gemma reads with causal attention (each token sees
only earlier tokens), so a pre-boundary signal can only come from foreshadowing
already present in the words read so far. Gemma's own stories foreshadow;
DeepSeek's cleaner transitions do not. The constant-emotion control's leads sit
at or below zero, so marked scene changes alone produce nothing.


In [3]:
# this cell draws the identity / anticipation dissociation heatmaps,
# story sets x probe banks, with a layer slider (default: layer 33)
banks = ["corpus", "selfgen", "deepseek"]
DEFAULT_LAYER = PRIMARY_LAYER

def dissociation_grids(layer):
    """(identity z, anticipation z, and their cell labels) at one layer."""
    identity, anticipation, id_text, ant_text = [], [], [], []
    for substrate, arm in SUBSTRATES.items():
        id_row, ant_row, id_t, ant_t = [], [], [], []
        for bank in banks:
            gate_cell = arm["gate_G"].get(bank, {}).get("per_layer", {}).get(str(layer))
            r1_cell = arm["r1_anticipation"].get(bank, {}).get("per_layer", {}).get(str(layer))
            if gate_cell:
                star = " *" if gate_cell["passes"] else ""
                id_row.append(gate_cell["median_rank"] / BANK_SIZE[bank])
                id_t.append(f"rank {gate_cell['median_rank']:.0f}/{BANK_SIZE[bank]}{star}")
            else:
                id_row.append(None); id_t.append("n/a")
            if r1_cell and r1_cell.get("noise_sd"):
                lead_in_sd = r1_cell["mean_lead"] / r1_cell["noise_sd"]
                star = " *" if r1_cell["passes"] else ""
                ant_row.append(lead_in_sd)
                ant_t.append(f"{lead_in_sd:.1f} sd{star}")
            else:
                ant_row.append(None); ant_t.append("n/a")
        identity.append(id_row); anticipation.append(ant_row)
        id_text.append(id_t); ant_text.append(ant_t)
    return identity, anticipation, id_text, ant_text

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.16,
    subplot_titles=(
        "identity read (gate G): rank / bank size (low = good)",
        "anticipation read (R1): pre-boundary lead, in noise-sd units"))
# one heatmap pair per layer; the slider toggles which pair is visible
for layer in LAYERS:
    identity, anticipation, id_text, ant_text = dissociation_grids(layer)
    visible = layer == DEFAULT_LAYER
    fig.add_trace(go.Heatmap(
        z=identity, x=[BANK_LABEL[b] for b in banks], y=list(SUBSTRATES),
        text=id_text, texttemplate="%{text}", colorscale="Blues_r", zmin=0, zmax=0.6,
        colorbar=dict(title=dict(text="rank / bank size", side="right"),
                      x=0.42, len=0.75, thickness=12),
        visible=visible), row=1, col=1)
    fig.add_trace(go.Heatmap(
        z=anticipation, x=[BANK_LABEL[b] for b in banks], y=list(SUBSTRATES),
        text=ant_text, texttemplate="%{text}", colorscale="RdBu", zmid=0,
        colorbar=dict(title=dict(text="lead (noise-sd)", side="right"),
                      x=1.0, len=0.75, thickness=12),
        visible=visible), row=1, col=2)
steps = []
for layer_pos, layer in enumerate(LAYERS):
    visibility = [other_pos == layer_pos for other_pos in range(len(LAYERS)) for _ in (0, 1)]
    steps.append(dict(method="update", label=f"layer {layer}", args=[{"visible": visibility}]))
# the title's headline numbers: self-gen lead on Gemma-written vs best lead on DeepSeek-written
gemma_lead_sd = (lambda c: c["mean_lead"] / c["noise_sd"])(
    SUBSTRATES["Gemma-written (v1 probes)"]["r1_anticipation"]["selfgen"]["per_layer"][str(DEFAULT_LAYER)])
deepseek_arm_best_sd = max(
    cell["mean_lead"] / cell["noise_sd"]
    for bank in banks
    if (cell := SUBSTRATES["DeepSeek-written stories"]["r1_anticipation"]
        .get(bank, {}).get("per_layer", {}).get(str(DEFAULT_LAYER)))
)
for annotation in fig.layout.annotations:  # only the panel titles exist at this point
    annotation.font = dict(size=13)
fig.update_layout(
    sliders=[dict(active=LAYERS.index(DEFAULT_LAYER), steps=steps, y=-0.22,
                  currentvalue=dict(prefix="showing: "))],
    title=(
        "Identity follows the probe bank; anticipation follows who wrote the stories<br>"
        f"<sup>at layer {DEFAULT_LAYER}, the self-gen bank's pre-boundary lead is "
        f"{gemma_lead_sd:.1f}x noise on Gemma-written stories but {deepseek_arm_best_sd:.1f}x "
        "(no pass) on DeepSeek-written ones</sup><br>"
        f"<sup>rows = story sets, columns = probe banks; * = passes its registered bar; "
        f"layer slider below | {MODEL}</sup>"
    ),
    title_font_size=15,
    width=1250, height=500, margin=dict(t=120, b=130))
fig.show()


<details><summary><b>How to read this figure</b></summary>

Both panels share the same grid: each row is one story set, each column one
probe bank, at the layer the slider selects. Left panel: the identity read.
The printed number is the tagged emotion's median rank over its bank size;
darker blue means nearer the top of the bank, which is better; the colorbar
between the panels gives the scale. Right panel: the anticipation read. The
printed number is the incoming emotion's pre-boundary lead in noise-sd units;
blue and positive means the incoming emotion is already rising before the
boundary, red and negative means it is falling; the right colorbar gives the
scale. "n/a" means that bank was not carried in that arm's shards (the
DeepSeek bank did not exist when the v1 Gemma-written arm was extracted). A
star marks a cell that passes its registered bar. To see the dissociation:
scan any COLUMN down the left panel (the corpus bank fails on every story set,
the 12-probe banks succeed on every story set), then scan the ROWS of the
right panel (only the Gemma-written rows show starred anticipation).

</details>

**What section 2 means.** Identity is a property of the probes: the same
column succeeds or fails regardless of who wrote the stories. Anticipation is
a property of the text: the same probes that pass on Gemma-written stories
show nothing on DeepSeek-written ones, and the constant-emotion control rules
out scene-change mechanics. The v2 row replicates the v1 row with post-fix
probes, so the anticipation result is not an artifact of the pre-fix corpus
extraction. Hypothesis (labeled as such): the pre-boundary rise is the model
detecting the foreshadowing habits of its own writing style; the deciding
experiment is the cue-referenced anticipation read (section 5, point 2).


## 3. Above chance versus useful versus meaningless

**Answer: three different situations, and we can tell them apart because
chance is measured, not assumed.** The wrong-emotion shuffle (N2) locates
chance: reassign the emotion labels at random 10,000 times and a meaningless
assignment lands at a median rank near half the bank size. The random
directions (N1, 24 of them) locate what meaningless vectors look like. The
instrument calibration measured the per-token noise floor, so a null result
comes with its detection power attached.

**How it was measured.** Every (story set, probe bank) identity result from
section 1 becomes one dot at the slider's layer. Its position on the
rank-fraction axis shows USEFULNESS (three shaded zones: past the registered
bar, above chance but short of the bar, chance-like); the printed shuffle p
shows DISTINGUISHABILITY from chance. These are different questions: with
thousands of phases, a median only modestly better than chance can still have
a tiny p, which is exactly the "bad but not meaningless" case. The corpus-171
bank lands squarely there: real signal (tiny shuffle p on the large story
arms; each dot prints its own p), not useful by the registered standard.

**Verdict.** Nothing we measured behaves like meaningless vectors. The corpus
bank is "bad but not meaningless". The self-gen and DeepSeek banks do good
work on identity in absolute terms (median rank 1 to 6 of 12), although on
the strict fraction axis only rank 1 of 12 clears the 0.1 bar (section 5,
point 4). For anticipation, bars are cleared only on Gemma-written stories
(the self-gen bank most strongly; the corpus bank weakly at three of six
layers; see the section 2 heatmap), and the constant-emotion control sits at
or below zero, so the surviving effect is not a scene-change artifact.


In [4]:
# this cell places every (story set, probe bank) identity result in the
# meaningless / above-chance / good-work zones, with a layer slider
fig = go.Figure()
symbol_cycle = ["circle", "square", "diamond", "triangle-up"]
marker_symbols = {substrate: symbol_cycle[i % len(symbol_cycle)]
                  for i, substrate in enumerate(SUBSTRATES)}
colors = {"corpus": "#7f7f7f", "selfgen": "#1f77b4", "deepseek": "#d62728"}
DEFAULT_LAYER = PRIMARY_LAYER

# one trace per (layer, story set, bank); only the active layer is visible
traces_per_layer = 0
fractions_by_layer = {layer: [] for layer in LAYERS}  # feeds each layer's title verdict
for layer in LAYERS:
    for substrate, arm in SUBSTRATES.items():
        for bank in ["corpus", "selfgen", "deepseek"]:
            gate_cell = arm["gate_G"].get(bank, {}).get("per_layer", {}).get(str(layer))
            if not gate_cell:
                continue
            if layer == LAYERS[0]:
                traces_per_layer += 1
            fraction = gate_cell["median_rank"] / BANK_SIZE[bank]
            fractions_by_layer[layer].append(fraction)
            fig.add_scatter(
                x=[fraction], y=[f"{BANK_LABEL[bank]}<br>{substrate}"],
                mode="markers+text", visible=(layer == DEFAULT_LAYER),
                marker=dict(size=14, color=colors[bank], symbol=marker_symbols[substrate]),
                text=[f"  rank {gate_cell['median_rank']:.0f}/{BANK_SIZE[bank]}, p={gate_cell['n2_p']:.4f}"],
                textposition="middle right", showlegend=False)

# per-layer title: the verdict must track the slider, never go stale
def zone_title(layer):
    fractions = fractions_by_layer[layer]
    n_chance_like = sum(fraction >= 0.48 for fraction in fractions)
    n_past_bar = sum(fraction <= 0.1 for fraction in fractions)
    return (
        f"Is any identity reading meaningless, and is any useful by the registered bar? "
        f"At layer {layer}: {n_chance_like} of {len(fractions)} chance-like, "
        f"{n_past_bar} past the bar<br>"
        "<sup>the rest sit above chance but short of the bar; printed p = probability that "
        "a wrong-emotion shuffle (N2, 10,000 shuffles) matches that median rank</sup><br>"
        f"<sup>one dot = one (story set, probe bank) at the slider's layer; "
        f"x = median rank / bank size, lower is better | {MODEL}</sup>"
    )


# slider: reveal exactly the chosen layer's traces AND retitle with its verdict
steps = []
for layer_pos, layer in enumerate(LAYERS):
    visibility = []
    for other_pos in range(len(LAYERS)):
        visibility += [other_pos == layer_pos] * traces_per_layer
    steps.append(dict(method="update", label=f"layer {layer}",
                      args=[{"visible": visibility}, {"title.text": zone_title(layer)}]))
fig.update_layout(sliders=[dict(
    active=LAYERS.index(DEFAULT_LAYER), steps=steps, y=-0.12,
    currentvalue=dict(prefix="showing: "))])
fig.add_vline(x=0.5, line_dash="dot", annotation_text="chance (N2 shuffle)",
              annotation_position="bottom left")
fig.add_vline(x=0.1, line_dash="dash", annotation_text="registered bar")
fig.add_vrect(x0=0.0, x1=0.1, fillcolor="green", opacity=0.06, line_width=0,
              annotation_text="doing good work", annotation_position="bottom left")
fig.add_vrect(x0=0.1, x1=0.48, fillcolor="orange", opacity=0.06, line_width=0,
              annotation_text="above chance, below the bar", annotation_position="bottom")
fig.add_vrect(x0=0.48, x1=0.62, fillcolor="red", opacity=0.06, line_width=0,
              annotation_text="chance-like", annotation_position="top right")
fig.update_layout(
    title=zone_title(DEFAULT_LAYER),
    title_font_size=15,
    xaxis_title="median rank / bank size", xaxis_range=[0, 0.62],
    width=1150, height=560, margin=dict(t=110))
fig.show()


<details><summary><b>How to read this figure</b></summary>

The y axis lists every (probe bank, story set) pair. The x axis is the
identity read at the slider's layer: the tagged emotion's median rank divided
by bank size, lower is better. The dotted vertical line at 0.5 is measured
chance (where wrong-emotion shuffles land); the dashed line at 0.1 is the
registered usefulness bar. The three shaded zones name the three situations:
green (left of the bar) = doing good work; orange = above chance but short of
the bar; red (near the chance line) = indistinguishable from a meaningless
assignment. Dot color is the probe bank (same colors as section 1); dot shape
is the story set. The text beside each dot prints the absolute rank and the
shuffle p. A dot in the orange zone with a tiny p is "bad but not
meaningless": with thousands of phases, even a modest advantage over chance is
detectable. Note the strictness of the fraction axis for small banks: only
rank 1 of 12 clears the 0.1 bar, so 12-probe dots at ranks 2 to 6 sit in the
orange zone while beating chance decisively (section 5, point 4, flags this
for adjudication).

</details>

**What section 3 means.** The three-way diagnosis separates cleanly: no
reading is chance-like, the corpus-171 bank carries real but unusable signal
under the registered standard, and the 12-probe banks are the working
instruments. What was ruled out: "the probes mean nothing" (random directions
and shuffles both say otherwise). What stayed possible: that the corpus bank's
failure is a probe-quality problem rather than a model limitation, which is
exactly what the bank comparison in section 2 says, and that the strict
fraction bar under-credits small banks, which awaits adjudication.


## 4. The full design grid and the combined reading

Everything scored so far as one factorial table: probe banks down the rows,
story sets across the columns. Cell shorthand, plain words first: "identity"
is the phase-identity read (gate G), "anticipation" is the pre-boundary read
(R1); **pass** = clears the registered bar; **substance** = far above chance
with the bar adjudication still open; **fail / zero** = does not clear and
shows no effect; "x noise" = the lead in units of the calibrated per-token
noise.

<table>
<tr><th>probe bank vs story set</th><th>Gemma-written (v1 probes)</th><th>Gemma-written (v2 post-fix probes)</th><th>DeepSeek-written</th><th>constant control</th></tr>
<tr><td><b>corpus-171</b></td><td>identity fail; anticipation weak pass (3 of 6 layers)</td><td>identity fail (robust to the post-fix probes); anticipation fail</td><td>identity fail; anticipation zero</td><td>identity fail; anticipation zero</td></tr>
<tr><td><b>self-gen (12)</b></td><td>identity substance; anticipation PASS (5.5x noise)</td><td>identity substance; anticipation PASS (lead 0.0117 vs 0.0116 in v1: replicates)</td><td>identity substance; anticipation zero</td><td>identity substance; anticipation zero</td></tr>
<tr><td><b>DeepSeek (12)</b></td><td>bank absent in the v1 shards</td><td>identity substance (rank 2/12); anticipation PASS (4.3x noise)</td><td>identity substance; anticipation zero</td><td>identity substance; anticipation zero</td></tr>
<tr><td><b>random (24)</b></td><td>null band</td><td>null band</td><td>null band</td><td>null band</td></tr>
</table>

The base-model arm is collected but unscored (reserved for the registered
falsify plan). Every entry above is readable off the section 1 and section 2
figures; the sd multiples and the v1-vs-v2 leads are printed in the section 2
heatmap cells.

**The combined reading.**

1. Whether the model's emotional state is readable at all is decided by the
   probe COLUMN: vector quality decides (the detection winners of
   [notebook 07](07_generator_lineages.ipynb), E10/E11, are the dynamics
   winners too), on every story set, and the corpus-171 failure survives the
   post-fix re-extraction.
2. Whether the state moves BEFORE a transition is decided by the story ROW:
   the same probes show anticipation exactly and only where Gemma authored the
   text; the control row rules out scene-change mechanics, and causal
   attention permits only foreshadowing already present in the words read so
   far.
3. Together: Gemma robustly tracks the emotional state of what it reads, and
   "seeing the transition coming" is the model detecting the foreshadowing
   habits of its own writing style. That makes anticipation a property of the
   MATCH between text and reader, not of the reader alone. No verdict-ladder
   tier is claimed until the ladder adjudication and the cue-referenced read
   (section 5).


## 5. What is still open

1. **Ladder adjudication.** The registered gate names the corpus-171 bank; it
   fails while the 12-probe banks pass in substance. The verdict-ladder tier
   (T0 through T3) needs that adjudicated before any claim graduates.
2. **The cue-referenced anticipation twin.** The collapse of anticipation on
   the DeepSeek-written arm says the surviving effect is textual foreshadowing.
   The decisive registered read re-references the anticipation window to
   judge-located cue positions (judges blind to activations); it needs the
   judge bulk pass.
3. **Ramp width and crossover location** (registry names R2 and R3) are
   registered and validated but not yet run.
4. **The small-bank bar is strict.** The 12-bank top-decile bar (rank 1 of 12
   or better) is the scorer's mechanical generalization of a bar registered
   for the 171-probe bank; median ranks of 2 to 6 of 12 fail it while beating
   the shuffle chance floor at nearly every layer (section 1 prints the ranks
   and worst shuffle p values). To be adjudicated alongside the ladder.
5. **Probe-version caveat (registry E4b).** The corpus-171 and self-gen dots
   in the v1 shards were computed against pre-padding-fix probe vectors. E4b
   rated the self-gen contrasts tier 1 (post-fix twins agree at about
   cos 0.995; numbers reproduce to roughly 3 decimals), and the v2 arm
   confirms the replication directly. The corpus-lineage contrasts were rated
   tier 2, so the v1 corpus-171 column carries a rescore-or-recollect caveat;
   the v2 column IS that rescore, and the identity failure survives it. The
   fixed-DeepSeek bank is post-fix by construction.

Nothing in this notebook is a graduated claim; the falsify gate
(shuffled-sentence control, category splits, base-arm replication) comes
before any of it enters a deliverable.
